## Retry and Delay

In [1]:
import os
import time
import logging
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage

from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("api_key")

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Initialize LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", groq_api_key=api_key)

def call_llm_with_retry(prompt, retries=3, delay=2):
    for attempt in range(1, retries + 1):
        try:
            logging.info(f"Attempt {attempt} - Calling LLM")
            
            response = llm.invoke([HumanMessage(content=prompt)])
            
            logging.info("LLM call successful")
            return response.content
        
        except Exception as e:
            logging.error(f"Attempt {attempt} failed: {e}")
            
            if attempt < retries:
                time.sleep(delay)
            else:
                logging.critical("All retry attempts failed")
                raise

# Example usage
if __name__ == "__main__":
    prompt = "Explain what is an API in one sentence"
    
    result = call_llm_with_retry(prompt)
    print("Final Response:", result)

/Users/ishant162/miniconda3/envs/rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-17 10:51:20,750 - INFO - Attempt 1 - Calling LLM
2026-04-17 10:51:21,145 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-04-17 10:51:21,162 - INFO - LLM call successful


Final Response: An API, or Application Programming Interface, is a set of defined rules and protocols that allows different software systems, applications, or services to communicate and exchange data with each other in a standardized and secure manner.


## Simple RAG

In [2]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
from pathlib import Path
import uuid
import chromadb
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

# ── Config ────────────────────────────────────────────────────────────────────
PDF_DIR        = "./data/"
VECTOR_DIR     = "./data/vector_store"
EMBED_MODEL    = "all-MiniLM-L6-v2"
CHUNK_SIZE     = 1000
CHUNK_OVERLAP  = 200
TOP_K          = 3

# ── 1. Load PDFs ──────────────────────────────────────────────────────────────
docs = []
for pdf in Path(PDF_DIR).glob("**/*.pdf"):
    docs.extend(PyPDFLoader(str(pdf)).load())
print(f"Loaded {len(docs)} pages")

# ── 2.1 Fixed Chunk ──────────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} chunks")

# ── 2.2 Semantic Chunk ──────────────────────────────────────────────────────────────────

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=70
)
chunks = splitter.split_documents(docs)
print(f"Split into {len(chunks)} semantic chunks")

# ── 3. Embed ──────────────────────────────────────────────────────────────────
model = SentenceTransformer(EMBED_MODEL)
texts = [c.page_content for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True)

# ── 4. Store in ChromaDB ──────────────────────────────────────────────────────
os.makedirs(VECTOR_DIR, exist_ok=True)

# Check write access
if not os.access(VECTOR_DIR, os.W_OK):
    raise PermissionError(f"No write permission for {VECTOR_DIR}")

client     = chromadb.PersistentClient(path=VECTOR_DIR)
collection = client.get_or_create_collection("pdf_documents")

collection.add(
    ids        = [f"doc_{uuid.uuid4().hex[:8]}_{i}" for i in range(len(chunks))],
    embeddings = embeddings.tolist(),
    documents  = texts,
    metadatas  = [c.metadata for c in chunks]
)
print(f"Stored {collection.count()} chunks in vector store")


2026-04-17 10:51:24,020 - INFO - Use pytorch device_name: mps
2026-04-17 10:51:24,020 - INFO - Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Loaded 15 pages
Split into 52 chunks


2026-04-17 10:51:29,370 - INFO - Use pytorch device_name: mps
2026-04-17 10:51:29,371 - INFO - Load pretrained SentenceTransformer: all-MiniLM-L6-v2


Split into 128 semantic chunks


Batches: 100%|██████████| 4/4 [00:00<00:00,  9.67it/s]
2026-04-17 10:51:33,548 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


Stored 128 chunks in vector store


In [3]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ── Shared setup (reuse model + collection from the main pipeline) ─────────────
texts        = [c.page_content for c in chunks]   # already have this from pipeline
tokenized    = [t.lower().split() for t in texts]
bm25         = BM25Okapi(tokenized)
reranker     = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# ── BM25 search ───────────────────────────────────────────────────────────────
def bm25_search(query, top_k):
    scores     = bm25.get_scores(query.lower().split())
    top_idx    = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [{"id": f"bm25_{i}", "content": texts[i], "score": scores[i]} for i in top_idx if scores[i] > 0]

# ── Vector search ─────────────────────────────────────────────────────────────
def vector_search(query, top_k):
    q_emb   = model.encode([query])[0].tolist()   # reuse SentenceTransformer from pipeline
    results = collection.query(query_embeddings=[q_emb], n_results=top_k)
    return [
        {"id": doc_id, "content": doc, "score": 1 - dist}
        for doc_id, doc, dist in zip(results["ids"][0], results["documents"][0], results["distances"][0])
    ]

# ── Reciprocal Rank Fusion ────────────────────────────────────────────────────
def rrf(bm25_results, vector_results, k=60):
    scores, doc_map = {}, {}
    for rank, doc in enumerate(bm25_results + vector_results, 1):
        scores[doc["id"]]  = scores.get(doc["id"], 0) + 1 / (k + rank)
        doc_map[doc["id"]] = doc
    return [doc_map[i] for i in sorted(scores, key=lambda x: scores[x], reverse=True)]

# ── RAG variants ──────────────────────────────────────────────────────────────
def rag(query, top_k=3):
    """Plain vector RAG"""
    results = vector_search(query, top_k)
    return _generate(query, results)

def rag_hybrid(query, top_k=3):
    """BM25 + Vector fused via RRF"""
    results = rrf(bm25_search(query, top_k * 3), vector_search(query, top_k * 3))[:top_k]
    return _generate(query, results)

def rag_hybrid_rerank(query, top_k=3):
    """BM25 + Vector → RRF → CrossEncoder rerank"""
    candidates = rrf(bm25_search(query, top_k * 4), vector_search(query, top_k * 4))[:top_k * 2]
    scores     = reranker.predict([(query, d["content"]) for d in candidates])
    results    = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)[:top_k]
    return _generate(query, [d for d, _ in results])

def rag_hyde(query, top_k=3):
    """HyDE: generate hypothetical answer → use it for retrieval"""
    hypo    = llm.invoke([f"Write a concise answer to: {query}"]).content
    results = rrf(bm25_search(hypo, top_k * 3), vector_search(hypo, top_k * 3))[:top_k]
    return _generate(query, results)

# ── Shared answer generator ───────────────────────────────────────────────────
def _generate(query, results):
    context = "\n\n".join([d["content"] for d in results])
    if not context:
        return "No relevant context found."
    prompt = f"Use the context below to answer the question.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"
    return llm.invoke([prompt]).content

# ── Run ───────────────────────────────────────────────────────────────────────
q = "What is attention is all you need?"

print("*"*50)
print("Basic RAG")
print(rag(q))
print("*"*50)
print("Hybrid RAG")
print(rag_hybrid(q))
print("*"*50)
print("Hybrid RAG rerank")
print(rag_hybrid_rerank(q))
print("*"*50)
print("Hyde RAG")
print(rag_hyde(q))
print("*"*50)

2026-04-17 10:51:38,863 - INFO - Use pytorch device: mps


**************************************************
Basic RAG


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.10it/s]
2026-04-17 10:51:40,380 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The context does not explicitly state what "Attention is All You Need" refers to. However, based on the topic of the passage, which discusses attention-based models, particularly the Transformer, it can be inferred that "Attention is All You Need" is likely the title of a research paper or a concept that emphasizes the importance of attention mechanisms in deep learning models, such as the Transformer. This paper, "Attention Is All You Need" by Vaswani et al., introduced the Transformer model, which relies entirely on attention mechanisms to process input sequences, eliminating the need for recurrent neural networks (RNNs) or convolutional neural networks (CNNs).
**************************************************
Hybrid RAG


Batches: 100%|██████████| 1/1 [00:00<00:00, 54.40it/s]
2026-04-17 10:51:40,972 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


"Attention Is All You Need" is the title of a research paper by Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, and Illia Polosukhin. The paper proposes a new sequence transduction model that relies entirely on attention mechanisms, eliminating the need for complex recurrent or convolutional neural networks. In other words, the paper argues that attention mechanisms are sufficient for sequence transduction tasks, and that they can replace traditional encoder-decoder architectures.
**************************************************
Hybrid RAG rerank


Batches: 100%|██████████| 1/1 [00:00<00:00, 23.72it/s]
2026-04-17 10:51:41,709 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


"Attention Is All You Need" is the title of a research paper by Ashish Vaswani et al., which introduces the Transformer model, a type of neural network architecture that relies heavily on self-attention mechanisms to process input sequences, eliminating the need for recurrent or convolutional neural networks. In essence, the paper argues that attention mechanisms are sufficient to handle complex sequence transduction tasks, such as machine translation, and can outperform traditional models.
**************************************************
Hyde RAG


2026-04-17 10:51:42,325 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Batches: 100%|██████████| 1/1 [00:00<00:00,  3.62it/s]
2026-04-17 10:51:43,349 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


"Attention Is All You Need" is the title of the paper that introduced the Transformer model, which relies entirely on self-attention mechanisms to perform sequence transduction tasks, such as machine translation and text generation, without using recurrent neural networks (RNNs) or convolutional neural networks (CNNs). The paper presents a new simple network architecture, the Transformer, that uses multi-headed self-attention to achieve state-of-the-art results in various tasks, including machine translation and English constituency parsing. The Transformer model is able to generalize well to other tasks and requires significantly less time to train than traditional RNN-based models.
**************************************************
